# Kaggle Competition — RiskScore Prediction

### Цель: построить модель линейной регрессии для предсказания RiskScore и оценить качество по MSE.

1. Импорт библиотек и загрузка данных

In [141]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("ex.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample_submission.shape)

display(train.head())
display(test.head())

Train: (11017, 35)
Test: (5000, 35)
Sample submission: (5000, 2)


,ApplicationDate,Age,AnnualIncome,CreditScore,LoanAmount,LoanDuration,MaritalStatus,NumberOfDependents,HomeOwnershipStatus,MonthlyDebtPayments,...,JobTenure,EmploymentStatus,EducationLevel,Experience,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio,RiskScore
0,2010-06-26,27.0,66829.0,549.0,17290.0,60.0,Divorced,1.0,Rent,1095.0,...,4.0,Employed,Associate,4.0,35067.0,0.257790,0.251465,508.970230,0.288013,66.176500
1,1996-09-23,55.0,172147.0,850.0,16110.0,36.0,Widowed,1.0,Mortgage,211.0,...,2.0,Employed,High School,33.0,27001.0,0.086110,0.093173,514.675859,0.050585,28.495737
2,2015-01-19,51.0,300000.0,850.0,38436.0,36.0,Married,0.0,Mortgage,546.0,...,3.0,Employed,Bachelor,28.0,278382.0,0.108436,0.115443,1268.276385,0.072571,34.488104
3,1981-05-12,25.0,34683.0,847.0,19186.0,48.0,Married,0.0,Other,153.0,...,3.0,Employed,High School,0.0,9224.0,0.100686,0.112822,498.505187,0.225415,36.910753
4,1995-05-07,55.0,300000.0,850.0,30437.0,48.0,Single,2.0,Rent,562.0,...,5.0,Employed,Bachelor,31.0,4502.0,0.110437,0.089037,756.035156,0.052721,31.347091


,ID,ApplicationDate,Age,AnnualIncome,CreditScore,LoanAmount,LoanDuration,MaritalStatus,NumberOfDependents,HomeOwnershipStatus,...,UtilityBillsPaymentHistory,JobTenure,EmploymentStatus,EducationLevel,Experience,NetWorth,BaseInterestRate,InterestRate,MonthlyLoanPayment,TotalDebtToIncomeRatio
0,0,1979-01-03,49,207095,850,14458,60,Single,0,Rent,...,0.883232,6,Employed,High School,23,10626,0.104458,0.125350,325.532336,0.040650
1,1,2006-04-10,28,64653,674,65491,48,Single,2,Own,...,0.791928,3,Employed,High School,3,299024,0.233491,0.260561,2210.247757,0.454410
2,2,2013-11-28,41,300000,768,16811,60,Single,1,Own,...,0.805826,3,Employed,High School,19,6979,0.147811,0.154571,403.977431,0.034919
3,3,1995-08-24,50,300000,850,48737,36,Divorced,0,Mortgage,...,0.829294,3,Self-Employed,Bachelor,26,8438,0.118737,0.099712,1571.946906,0.085558
4,4,1980-06-15,35,300000,672,51916,60,Single,6,Mortgage,...,0.735013,7,Employed,Associate,12,436216,0.230916,0.237727,1486.675235,0.063707


2. Проверка и очистка целевой переменной

In [142]:
print("Пропуски в RiskScore:", train["RiskScore"].isna().sum())
print("Маркер -9999999:", (train["RiskScore"] == -9999999).sum())
print("Маркер 10000000:", (train["RiskScore"] == 10000000).sum())

train_model = train[
    train["RiskScore"].notna()
    & train["RiskScore"].between(0, 100)
].copy()

print("Размер исходного train:", train.shape)
print("Размер train_model:", train_model.shape)
display(train_model["RiskScore"].describe())

Пропуски в RiskScore: 530
Маркер -9999999: 121
Маркер 10000000: 94
Размер исходного train: (11017, 35)
Размер train_model: (10272, 35)


count    10272.000000
mean        48.358094
std         17.241456
min         14.841417
25%         32.722756
50%         44.195792
75%         65.124190
max         97.597249
Name: RiskScore, dtype: float64

3. Разделение на обучающую и проверочную выборки

In [143]:
X = train_model.drop(columns=["RiskScore"])
y = train_model["RiskScore"].astype(float)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

X_train: (8217, 34)
X_val: (2055, 34)


4. Подготовка признаков

In [144]:
def add_date_features(df):
    df = df.copy()
    date = pd.to_datetime(df["ApplicationDate"], errors="coerce")
    df["ApplicationYear"] = date.dt.year
    df["ApplicationMonth"] = date.dt.month
    df["ApplicationDay"] = date.dt.day
    return df.drop(columns=["ApplicationDate"])

X_train = add_date_features(X_train)
X_val = add_date_features(X_val)

categorical_features = X_train.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

print("Числовых признаков:", len(numeric_features))
print("Категориальные признаки:", categorical_features)

Числовых признаков: 31
Категориальные признаки: ['MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose', 'EmploymentStatus', 'EducationLevel']


Заполнение пропусков

In [145]:
numeric_medians = X_train[numeric_features].median()
categorical_modes = X_train[categorical_features].mode().iloc[0]

X_train[numeric_features] = X_train[numeric_features].fillna(numeric_medians)
X_val[numeric_features] = X_val[numeric_features].fillna(numeric_medians)

X_train[categorical_features] = X_train[categorical_features].fillna(categorical_modes)
X_val[categorical_features] = X_val[categorical_features].fillna(categorical_modes)

print("Пропуски в X_train:", X_train.isna().sum().sum())
print("Пропуски в X_val:", X_val.isna().sum().sum())

Пропуски в X_train: 0
Пропуски в X_val: 0


In [304]:
categorical_cols = train_model.select_dtypes(
    include=["object", "category"]
).columns

for col in categorical_cols:
    print(f"\n{col}")
    print("Количество категорий:", train_model[col].nunique())
    print(train_model[col].value_counts().head(10))


ApplicationDate
Количество категорий: 9788
ApplicationDate
1988-12-02    3
1986-05-04    3
2004-03-27    3
1995-04-25    3
1983-01-31    3
1996-08-15    3
1980-04-06    3
1996-04-03    3
2011-04-20    3
2016-03-23    3
Name: count, dtype: int64

MaritalStatus
Количество категорий: 4
MaritalStatus
Married     5023
Single      3163
Divorced    1536
Widowed      550
Name: count, dtype: int64

HomeOwnershipStatus
Количество категорий: 4
HomeOwnershipStatus
Mortgage    4057
Rent        3115
Own         2081
Other       1019
Name: count, dtype: int64

LoanPurpose
Количество категорий: 5
LoanPurpose
Home                  2919
Debt Consolidation    2407
Auto                  1968
Education             1573
Other                  920
Name: count, dtype: int64

EmploymentStatus
Количество категорий: 3
EmploymentStatus
Employed         8884
Self-Employed     966
Unemployed        422
Name: count, dtype: int64

EducationLevel
Количество категорий: 5
EducationLevel
High School    4150
Bachelor    

C:\Users\User\AppData\Local\Temp\ipykernel_12052\1703682083.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train_model.select_dtypes(


One-Hot Encoding

In [146]:
X_train = pd.get_dummies(X_train, columns=categorical_features, dtype=int)
X_val = pd.get_dummies(X_val, columns=categorical_features, dtype=int)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Размер X_train:", X_train.shape)
print("Размер X_val:", X_val.shape)

Размер X_train: (8217, 52)
Размер X_val: (2055, 52)


Доп рефакторинг

In [299]:
for frame in [X_train, X_val]:

    # Нелинейное влияние дохода
    frame["Log_MonthlyIncome"] = np.log1p(
        frame["MonthlyIncome"].clip(lower=0)
    )
    frame["Log_MonthlyIncome_sq"] = frame["Log_MonthlyIncome"] ** 2

    # Нелинейное влияние кредитного рейтинга
    frame["CreditScore_sq"] = frame["CreditScore"] ** 2

    # Нелинейное влияние активов
    frame["Log_TotalAssets"] = np.log1p(
        frame["TotalAssets"].clip(lower=0)
    )

    # Нелинейное влияние базовой процентной ставки
    frame["BaseInterestRate_sq"] = (
        frame["BaseInterestRate"] ** 2
    )

    # Нелинейное влияние долговой нагрузки
    frame["Log_TotalDebtToIncomeRatio"] = np.log1p(
        frame["TotalDebtToIncomeRatio"].clip(lower=0)
    )

    # Новые преобразования

    # Логарифм годового дохода
    frame["Log_AnnualIncome"] = np.log1p(
        frame["AnnualIncome"].clip(lower=0)
    )

    # Квадрат процентной ставки
    frame["InterestRate_sq"] = (
        frame["InterestRate"] ** 2
    )

    # Квадрат долговой нагрузки
    frame["TotalDebtToIncomeRatio_sq"] = (
        frame["TotalDebtToIncomeRatio"] ** 2
    )
    frame["MonthlyIncome_sq"] = frame["MonthlyIncome"] ** 2
    frame["Log_CreditScore"] = np.log1p(frame["CreditScore"].clip(lower=0))
    
    frame["Log_CreditScore_sq"] = frame["Log_CreditScore"] ** 2

    frame["Log_InterestRate"] = np.log1p( frame["InterestRate"].clip(lower=0))
    frame["Log_InterestRate_sq"] = frame["Log_InterestRate"] ** 2
    frame["TotalAssets_sq"] = frame["TotalAssets"] ** 2
    frame["BaseInterestRate_cube"] = frame["BaseInterestRate"] ** 3
print("Количество признаков:", X_train.shape[1])
print("Train:", X_train.shape)
print("Validation:", X_val.shape)


Количество признаков: 72
Train: (8217, 72)
Validation: (2055, 72)


Z-score нормализация

In [300]:
z_mean = X_train.mean()
z_std = X_train.std(ddof=0).replace(0, 1)

X_train_z = ((X_train - z_mean) / z_std).astype(float)
X_val_z = ((X_val - z_mean) / z_std).astype(float)

print("Пропуски после нормализации:", X_train_z.isna().sum().sum())
print("Размеры:", X_train_z.shape, X_val_z.shape)

Пропуски после нормализации: 0
Размеры: (8217, 72) (2055, 72)


5. Собственная линейная регрессия

In [234]:
class MyLinearRegression:
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        X_bias = np.c_[np.ones(X.shape[0]), X]
        self.weights = np.linalg.pinv(X_bias) @ y
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_bias = np.c_[np.ones(X.shape[0]), X]
        return X_bias @ self.weights

model = MyLinearRegression()
model.fit(X_train_z, y_train)

y_pred = model.predict(X_val_z)

Базовые метрики

In [235]:
mse = mean_squared_error(y_val, y_pred)
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2:  {r2:.4f}")

MSE: 36.1133
MAE: 4.6774
R2:  0.8792


6. Анализ ошибок модели

In [ ]:
errors = pd.DataFrame({
    "Actual": y_val.to_numpy(),
    "Predicted": y_pred
})
errors["Error"] = errors["Actual"] - errors["Predicted"]
errors["AbsoluteError"] = errors["Error"].abs()
errors["SquaredError"] = errors["Error"] ** 2

print("Средняя ошибка:", errors["Error"].mean())
print("Медианная абсолютная ошибка:", errors["AbsoluteError"].median())
print("95-й процентиль абсолютной ошибки:", errors["AbsoluteError"].quantile(0.95))

display(errors.sort_values("AbsoluteError", ascending=False).head(15))

Фактические и предсказанные значения

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(errors["Actual"], errors["Predicted"], alpha=0.4)
plt.plot([0, 100], [0, 100], "r--")
plt.xlabel("Actual RiskScore")
plt.ylabel("Predicted RiskScore")
plt.title("Actual vs Predicted RiskScore")
plt.grid()
plt.show()

График остатков

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(errors["Actual"], errors["Error"], alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Actual RiskScore")
plt.ylabel("Prediction Error")
plt.title("Residual Plot")
plt.grid()
plt.show()

MSE по диапазонам фактического RiskScore

In [127]:
errors["ScoreGroup"] = pd.cut(
    errors["Actual"],
    bins=[0, 35, 65, 100],
    labels=["Низкий (0–35)", "Средний (35–65)", "Высокий (65–100)"],
    include_lowest=True
)

group_errors = errors.groupby("ScoreGroup", observed=False).agg(
    Count=("SquaredError", "count"),
    MSE=("SquaredError", "mean"),
    MeanActual=("Actual", "mean"),
    MeanPredicted=("Predicted", "mean")
)

display(group_errors)

,Count,MSE,MeanActual,MeanPredicted
ScoreGroup,,,,
Низкий (0–35),711,52.068240,30.008214,32.687375
Средний (35–65),829,53.595731,48.349628,48.862010
Высокий (65–100),515,56.731640,71.856773,68.061763


Анализ ошибок по группам целевой переменной показал, что модель имеет тенденцию завышать низкие значения RiskScore и занижать высокие. Наиболее высокая ошибка наблюдается в группе высоких значений (MSE = 56.73), а наименьшая — в группе низких значений (MSE = 52.07). Это может свидетельствовать о недостаточной способности линейной модели описывать зависимости на краях диапазона целевой переменной.

### 7. Выводы и дальнейшие эксперименты

ПРоверим взаимодействие

### Проверим устойчивость выборки 35.4696

5 разделений

In [301]:
best_pairs = [
    ("MonthlyIncome", "BaseInterestRate"),
    ("CreditScore", "MonthlyIncome"),
    ("CreditScore", "BaseInterestRate")
]

best_triple = (
    "CreditScore",
    "MonthlyIncome",
    "BaseInterestRate"
)

In [292]:
def my_kfold(n_samples, n_splits=5, shuffle=True, random_state=42):
    indices = np.arange(n_samples)

    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)

    folds = np.array_split(indices, n_splits)

    for i in range(n_splits):
        val_indices = folds[i]
        train_indices = np.concatenate(
            [folds[j] for j in range(n_splits) if j != i]
        )

        yield train_indices, val_indices


def my_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


In [293]:

fold_mse_scores = []

for fold_number, (train_idx, val_idx) in enumerate(
    my_kfold(len(X_train), n_splits=5),
    start=1
):
    # Разделяем данные на обучающую и проверочную части
    X_fold_train = X_train.iloc[train_idx].copy()
    X_fold_val = X_train.iloc[val_idx].copy()

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    # Стандартизация только по обучающей части фолда
    fold_mean = X_fold_train.mean()
    fold_std = X_fold_train.std(ddof=0).replace(0, 1)

    X_fold_train = (X_fold_train - fold_mean) / fold_std
    X_fold_val = (X_fold_val - fold_mean) / fold_std

    # Добавляем лучшие парные взаимодействия
    for feature_1, feature_2 in best_pairs:

        name = f"{feature_1}_x_{feature_2}"

        X_fold_train[name] = (
            X_fold_train[feature_1] *
            X_fold_train[feature_2]
        )

        X_fold_val[name] = (
            X_fold_val[feature_1] *
            X_fold_val[feature_2]
        )

        # Масштабируем взаимодействие по train-фолду
        mean = X_fold_train[name].mean()
        std = X_fold_train[name].std(ddof=0)

        if std == 0:
            std = 1

        X_fold_train[name] = (
            X_fold_train[name] - mean
        ) / std

        X_fold_val[name] = (
            X_fold_val[name] - mean
        ) / std

    # Добавляем лучшее тройное взаимодействие
    name = "_x_".join(best_triple)

    X_fold_train[name] = (
        X_fold_train[best_triple[0]] *
        X_fold_train[best_triple[1]] *
        X_fold_train[best_triple[2]]
    )

    X_fold_val[name] = (
        X_fold_val[best_triple[0]] *
        X_fold_val[best_triple[1]] *
        X_fold_val[best_triple[2]]
    )

    mean = X_fold_train[name].mean()
    std = X_fold_train[name].std(ddof=0)

    if std == 0:
        std = 1

    X_fold_train[name] = (
        X_fold_train[name] - mean
    ) / std

    X_fold_val[name] = (
        X_fold_val[name] - mean
    ) / std

    # Обучаем модель
    model = MyLinearRegression()
    model.fit(
        X_fold_train.to_numpy(dtype=float),
        y_fold_train.to_numpy(dtype=float)
    )

    predictions = model.predict(
        X_fold_val.to_numpy(dtype=float)
    )

    fold_mse = my_mse(
        y_fold_val.to_numpy(dtype=float),
        predictions
    )

    fold_mse_scores.append(fold_mse)

    print(f"Fold {fold_number} MSE: {fold_mse:.4f}")

print("\nСредний MSE:", np.mean(fold_mse_scores))
print("Стандартное отклонение MSE:", np.std(fold_mse_scores))


Fold 1 MSE: 31.0165
Fold 2 MSE: 28.1708
Fold 3 MSE: 30.0573
Fold 4 MSE: 32.5992
Fold 5 MSE: 30.0161

Средний MSE: 30.371969899640042
Стандартное отклонение MSE: 1.4454557644746824


In [302]:
class MyRidgeRegression:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.weights = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        # Добавляем столбец единиц для свободного коэффициента
        X_bias = np.c_[np.ones(X.shape[0]), X]

        # Матрица регуляризации
        penalty = np.eye(X_bias.shape[1])

        # Не штрафуем свободный коэффициент (intercept)
        penalty[0, 0] = 0

        # Аналитическое решение Ridge-регрессии
        self.weights = np.linalg.pinv(
            X_bias.T @ X_bias + self.alpha * penalty
        ) @ X_bias.T @ y

        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)

        X_bias = np.c_[np.ones(X.shape[0]), X]

        return X_bias @ self.weights


Значения в норме,значит такое изменение выборки дало хороший результат

In [306]:
alphas = [0.00001, 0.0001, 0.001, 0.005, 0.01, 0.02, 0.05]
ridge_results = []

for alpha in alphas:

    fold_scores = []

    for train_idx, val_idx in my_kfold(
        len(X_train), n_splits=5
    ):
        X_fold_train = X_train.iloc[train_idx].copy()
        X_fold_val = X_train.iloc[val_idx].copy()

        y_fold_train = y_train.iloc[train_idx].to_numpy(dtype=float)
        y_fold_val = y_train.iloc[val_idx].to_numpy(dtype=float)

        # Масштабируем исходные признаки
        mean = X_fold_train.mean()
        std = X_fold_train.std(ddof=0).replace(0, 1)

        X_fold_train = (X_fold_train - mean) / std
        X_fold_val = (X_fold_val - mean) / std

        # Добавляем три парных взаимодействия
        for f1, f2 in best_pairs:
            name = f"{f1}_x_{f2}"

            X_fold_train[name] = X_fold_train[f1] * X_fold_train[f2]
            X_fold_val[name] = X_fold_val[f1] * X_fold_val[f2]

            m = X_fold_train[name].mean()
            s = X_fold_train[name].std(ddof=0)

            if s == 0:
                s = 1

            X_fold_train[name] = (X_fold_train[name] - m) / s
            X_fold_val[name] = (X_fold_val[name] - m) / s

        # Добавляем тройное взаимодействие
        name = "_x_".join(best_triple)

        X_fold_train[name] = (
            X_fold_train[best_triple[0]]
            * X_fold_train[best_triple[1]]
            * X_fold_train[best_triple[2]]
        )

        X_fold_val[name] = (
            X_fold_val[best_triple[0]]
            * X_fold_val[best_triple[1]]
            * X_fold_val[best_triple[2]]
        )

        m = X_fold_train[name].mean()
        s = X_fold_train[name].std(ddof=0)

        if s == 0:
            s = 1

        X_fold_train[name] = (X_fold_train[name] - m) / s
        X_fold_val[name] = (X_fold_val[name] - m) / s

        # Обучаем Ridge
        model = MyRidgeRegression(alpha=alpha)

        model.fit(
            X_fold_train.to_numpy(dtype=float),
            y_fold_train
        )

        predictions = model.predict(
            X_fold_val.to_numpy(dtype=float)
        )

        fold_scores.append(my_mse(y_fold_val, predictions))

    ridge_results.append({
        "Alpha": alpha,
        "Mean MSE": np.mean(fold_scores),
        "Std MSE": np.std(fold_scores)
    })

display(pd.DataFrame(ridge_results))


,Alpha,Mean MSE,Std MSE
0,0.00001,30.333052,1.444280
1,0.00010,30.369035,1.441482
2,0.00100,30.405606,1.452676
3,0.00500,30.637797,1.478855
4,0.01000,30.830526,1.493851
5,0.02000,31.026441,1.509157
6,0.05000,31.236023,1.526669


Вывод, фиксируем альфа 0.005

Попробуем новые преобразования признаков

In [307]:
alpha = 0.00001
fold_scores = []

for train_idx, val_idx in my_kfold(len(X_train), n_splits=5):

    X_fold_train = X_train.iloc[train_idx].copy()
    X_fold_val = X_train.iloc[val_idx].copy()

    y_fold_train = y_train.iloc[train_idx].to_numpy(dtype=float)
    y_fold_val = y_train.iloc[val_idx].to_numpy(dtype=float)

    # Масштабируем исходные признаки
    mean = X_fold_train.mean()
    std = X_fold_train.std(ddof=0).replace(0, 1)

    X_fold_train = (X_fold_train - mean) / std
    X_fold_val = (X_fold_val - mean) / std

    # Обучаем Ridge
    model = MyRidgeRegression(alpha=alpha)

    model.fit(
        X_fold_train.to_numpy(dtype=float),
        y_fold_train
    )

    predictions = model.predict(
        X_fold_val.to_numpy(dtype=float)
    )

    fold_scores.append(my_mse(y_fold_val, predictions))

print("MSE каждого фолда:", fold_scores)
print("Средний MSE:", np.mean(fold_scores))
print("Стандартное отклонение MSE:", np.std(fold_scores))

MSE каждого фолда: [np.float64(31.219686597660026), np.float64(28.78487977131791), np.float64(30.247068154817047), np.float64(33.08572417200867), np.float64(30.378454622114962)]
Средний MSE: 30.743162663583725
Стандартное отклонение MSE: 1.4091602523335018


In [308]:
best_alpha = 0.00001  # замени, если в лучшем результате был другой alpha

X_array = np.asarray(X_train, dtype=float)
y_array = np.asarray(y_train, dtype=float)

scores_original = []
scores_log = []

for train_idx, val_idx in my_kfold(len(y_array), n_splits=5, random_state=42):

    X_tr = X_array[train_idx]
    X_val_fold = X_array[val_idx]

    y_tr = y_array[train_idx]
    y_val_fold = y_array[val_idx]

    # Масштабирование только по обучающей части фолда
    mean = X_tr.mean(axis=0)
    std = X_tr.std(axis=0)
    std[std == 0] = 1

    X_tr_scaled = (X_tr - mean) / std
    X_val_scaled = (X_val_fold - mean) / std

    # Вариант 1: исходная целевая переменная
    model_original = MyRidgeRegression(alpha=best_alpha)
    model_original.fit(X_tr_scaled, y_tr)

    pred_original = model_original.predict(X_val_scaled)
    scores_original.append(my_mse(y_val_fold, pred_original))

    # Вариант 2: логарифм целевой переменной
    model_log = MyRidgeRegression(alpha=best_alpha)
    model_log.fit(X_tr_scaled, np.log1p(y_tr))

    pred_log = model_log.predict(X_val_scaled)
    pred_log = np.expm1(pred_log)

    scores_log.append(my_mse(y_val_fold, pred_log))

print("Исходный RiskScore:")
print("MSE по фолдам:", scores_original)
print("Средний MSE:", np.mean(scores_original))

print("\nЛогарифмированный RiskScore:")
print("MSE по фолдам:", scores_log)
print("Средний MSE:", np.mean(scores_log))


Исходный RiskScore:
MSE по фолдам: [np.float64(31.21968753538479), np.float64(28.784878860609115), np.float64(30.24706770900436), np.float64(33.08572424433421), np.float64(30.378451262201118)]
Средний MSE: 30.74316192230672

Логарифмированный RiskScore:
MSE по фолдам: [np.float64(32.85401948243884), np.float64(29.404644304158644), np.float64(30.8754184374195), np.float64(33.03351714380629), np.float64(31.211035228149285)]
Средний MSE: 31.47572691919451


In [309]:

X_array = np.asarray(X_train, dtype=float)
y_array = np.asarray(y_train, dtype=float)

oof_predictions = np.zeros(len(y_array))

for train_idx, val_idx in my_kfold(
    len(y_array), n_splits=5, random_state=42
):
    X_tr = X_array[train_idx]
    X_val_fold = X_array[val_idx]

    y_tr = y_array[train_idx]

    mean = X_tr.mean(axis=0)
    std = X_tr.std(axis=0)
    std[std == 0] = 1

    X_tr_scaled = (X_tr - mean) / std
    X_val_scaled = (X_val_fold - mean) / std

    model = MyRidgeRegression(alpha=best_alpha)
    model.fit(X_tr_scaled, y_tr)

    oof_predictions[val_idx] = model.predict(X_val_scaled)

residuals = y_array - oof_predictions

print("OOF MSE:", my_mse(y_array, oof_predictions))
print("Средняя ошибка:", residuals.mean())
print("Средняя абсолютная ошибка:", np.mean(np.abs(residuals)))

# Связь остатков с признаками
residual_df = pd.DataFrame({
    "feature": X_train.columns,
    "residual_correlation": [
        np.corrcoef(X_train[col], residuals)[0, 1]
        for col in X_train.columns
    ]
})

residual_df["abs_correlation"] = (
    residual_df["residual_correlation"].abs()
)

display(
    residual_df.sort_values(
        "abs_correlation", ascending=False
    ).head(20)
)


OOF MSE: 30.74298159403014
Средняя ошибка: -0.00575442840234684
Средняя абсолютная ошибка: 4.235361722176368


c:\MyProjects\5_semester\mach_obuch\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\MyProjects\5_semester\mach_obuch\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3038: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,feature,residual_correlation,abs_correlation
67,TotalAssets_sq,-0.014460,0.014460
61,TotalDebtToIncomeRatio_sq,-0.007448,0.007448
17,TotalAssets,-0.007307,0.007307
23,NetWorth,-0.007202,0.007202
18,TotalLiabilities,-0.002851,0.002851
27,TotalDebtToIncomeRatio,-0.002668,0.002668
4,LoanDuration,-0.002526,0.002526
68,BaseInterestRate_cube,-0.001906,0.001906
64,AnnualIncome_sq,-0.001762,0.001762
21,JobTenure,0.001606,0.001606


In [312]:

important_features = [
    "CreditScore",
    "MonthlyIncome",
    "BaseInterestRate",
    "InterestRate",
    "TotalDebtToIncomeRatio"
]

output = []

for feature in important_features:
    temp = pd.DataFrame({
        "feature": X_train[feature].values,
        "residual": residuals
    })

    temp["group"] = pd.qcut(
        temp["feature"],
        q=10,
        duplicates="drop"
    )

    result = temp.groupby(
        "group", observed=True
    )["residual"].agg(["mean", "count"])

    output.append(f"\nПризнак: {feature}\n{result.round(3).to_string()}")

print("\n".join(output))



Признак: CreditScore
                   mean  count
group                         
(299.999, 408.0]  0.188    824
(408.0, 514.2]   -0.348    820
(514.2, 598.0]    0.360    827
(598.0, 669.0]   -0.296    830
(669.0, 720.0]    0.379   1002
(720.0, 771.0]   -0.614    632
(771.0, 843.0]    0.015    822
(843.0, 850.0]    0.011   2460

Признак: MonthlyIncome
                         mean  count
group                               
(1249.999, 2554.033]   -0.006   2465
(2554.033, 4645.917]   -0.051    822
(4645.917, 7282.417]    0.176    822
(7282.417, 11291.983]  -0.010    821
(11291.983, 17207.033] -0.332    822
(17207.033, 25000.0]    0.059   2465

Признак: BaseInterestRate
                  mean  count
group                        
(0.052, 0.0952]  0.090    822
(0.0952, 0.113]  0.022    822
(0.113, 0.135]  -0.040    821
(0.135, 0.159]  -0.363    822
(0.159, 0.184]   0.297   1015
(0.184, 0.207]  -0.174    628
(0.207, 0.242]  -0.060    822
(0.242, 0.284]   0.214    821
(0.284, 0.34]   -0.07

In [324]:
X_array = np.asarray(X_train, dtype=float)
y_array = np.asarray(y_train, dtype=float)

ratio_idx = X_train.columns.get_loc("TotalDebtToIncomeRatio")
interest_idx = X_train.columns.get_loc("InterestRate")
base_interest_idx = X_train.columns.get_loc("BaseInterestRate")
credit_idx = X_train.columns.get_loc("CreditScore")
income_idx = X_train.columns.get_loc("MonthlyIncome")
annual_income_idx = X_train.columns.get_loc("AnnualIncome")
assets_idx = X_train.columns.get_loc("TotalAssets")
age_idx = X_train.columns.get_loc("Age")

previous_scores = []
all_scores = []

for train_idx, val_idx in my_kfold(
    n_samples=len(y_array),
    n_splits=5,
    shuffle=True,
    random_state=42
):
    X_fold_train = X_array[train_idx]
    X_fold_val = X_array[val_idx]

    y_fold_train = y_array[train_idx]
    y_fold_val = y_array[val_idx]

    # Исходные значения семи признаков
    ratio_train, ratio_val = X_fold_train[:, ratio_idx], X_fold_val[:, ratio_idx]
    interest_train, interest_val = X_fold_train[:, interest_idx], X_fold_val[:, interest_idx]
    base_train, base_val = X_fold_train[:, base_interest_idx], X_fold_val[:, base_interest_idx]
    credit_train, credit_val = X_fold_train[:, credit_idx], X_fold_val[:, credit_idx]
    income_train, income_val = X_fold_train[:, income_idx], X_fold_val[:, income_idx]
    annual_train, annual_val = X_fold_train[:, annual_income_idx], X_fold_val[:, annual_income_idx]
    assets_train, assets_val = X_fold_train[:, assets_idx], X_fold_val[:, assets_idx]

    # Рассчитываем узлы только по обучающей части фолда
    knots = [
        np.quantile(values, [0.2, 0.4, 0.6, 0.8])
        for values in [
            ratio_train, interest_train, base_train, credit_train,
            income_train, annual_train, assets_train
        ]
    ]

    train_values = [
        ratio_train, interest_train, base_train, credit_train,
        income_train, annual_train, assets_train
    ]
    val_values = [
        ratio_val, interest_val, base_val, credit_val,
        income_val, annual_val, assets_val
    ]

    train_hinges = [
        np.column_stack([np.maximum(values - k, 0) for k in feature_knots])
        for values, feature_knots in zip(train_values, knots)
    ]

    val_hinges = [
        np.column_stack([np.maximum(values - k, 0) for k in feature_knots])
        for values, feature_knots in zip(val_values, knots)
    ]

    # Модель с семью преобразованиями
    X_previous_train = np.column_stack([X_fold_train] + train_hinges)
    X_previous_val = np.column_stack([X_fold_val] + val_hinges)

    mean_previous = X_previous_train.mean(axis=0)
    std_previous = X_previous_train.std(axis=0)
    std_previous[std_previous == 0] = 1

    X_previous_train_z = (X_previous_train - mean_previous) / std_previous
    X_previous_val_z = (X_previous_val - mean_previous) / std_previous

    model_previous = MyRidgeRegression(alpha=best_alpha)
    model_previous.fit(X_previous_train_z, y_fold_train)

    previous_pred = model_previous.predict(X_previous_val_z)
    previous_scores.append(my_mse(y_fold_val, previous_pred))

    # Добавляем кусочно-линейное преобразование Age
    age_train = X_fold_train[:, age_idx]
    age_val = X_fold_val[:, age_idx]

    age_knots = np.quantile(age_train, [0.2, 0.4, 0.6, 0.8])

    age_train_hinge = np.column_stack([
        np.maximum(age_train - k, 0) for k in age_knots
    ])

    age_val_hinge = np.column_stack([
        np.maximum(age_val - k, 0) for k in age_knots
    ])

    X_all_train = np.column_stack([X_previous_train, age_train_hinge])
    X_all_val = np.column_stack([X_previous_val, age_val_hinge])

    mean_all = X_all_train.mean(axis=0)
    std_all = X_all_train.std(axis=0)
    std_all[std_all == 0] = 1

    X_all_train_z = (X_all_train - mean_all) / std_all
    X_all_val_z = (X_all_val - mean_all) / std_all

    model_all = MyRidgeRegression(alpha=best_alpha)
    model_all.fit(X_all_train_z, y_fold_train)

    all_pred = model_all.predict(X_all_val_z)
    all_scores.append(my_mse(y_fold_val, all_pred))


print("Семь предыдущих преобразований:")
print("По фолдам:", np.round(previous_scores, 4))
print("Средний MSE:", round(np.mean(previous_scores), 5))

print("\nВсе восемь преобразований:")
print("По фолдам:", np.round(all_scores, 4))
print("Средний MSE:", round(np.mean(all_scores), 5))

Семь предыдущих преобразований:
По фолдам: [29.8669 25.779  28.464  29.7333 27.7111]
Средний MSE: 28.31085

Все восемь преобразований:
По фолдам: [29.8192 25.938  28.4236 29.6563 27.7317]
Средний MSE: 28.31376


In [ ]:
def add_piecewise_features(X, knots=None):
    X = X.copy()

    piecewise_columns = [
        "TotalDebtToIncomeRatio",
        "InterestRate",
        "BaseInterestRate",
        "CreditScore",
        "MonthlyIncome",
        "AnnualIncome",
        "TotalAssets"
    ]

    if knots is None:
        knots = {}

        for column in piecewise_columns:
            knots[column] = np.quantile(
                X[column],
                [0.2, 0.4, 0.6, 0.8]
            )

    for column in piecewise_columns:
        for i, knot in enumerate(knots[column]):
            X[f"{column}_hinge_{i+1}"] = np.maximum(
                X[column] - knot, 0
            )

    return X, knots


# Создаём преобразования для обучающей выборки
X_train_final, piecewise_knots = add_piecewise_features(X_train)

# Применяем те же узлы к проверочной выборке
X_val_final, _ = add_piecewise_features(
    X_val,
    knots=piecewise_knots
)

print("Исходное количество признаков:", X_train.shape[1])
print("Количество признаков после преобразования:", X_train_final.shape[1])
print("Добавлено кусочно-линейных признаков:", 
      X_train_final.shape[1] - X_train.shape[1])

Из эксперимента с кусочными функциями фиксируем лучший результат

In [325]:
BASE_PIECEWISE_FEATURES = [
    "TotalDebtToIncomeRatio",
    "InterestRate",
    "BaseInterestRate",
    "CreditScore",
    "MonthlyIncome",
    "AnnualIncome",
    "TotalAssets"
]

BASE_ALPHA = best_alpha

BASE_RANDOM_STATE = 42
BASE_N_SPLITS = 5

BASELINE_CV_MSE = 28.31085

print("Базовая конфигурация сохранена")
print("Количество кусочно-линейных признаков:",
      len(BASE_PIECEWISE_FEATURES))
print("Alpha:", BASE_ALPHA)
print("Количество фолдов:", BASE_N_SPLITS)
print("Контрольный CV MSE:", BASELINE_CV_MSE)

Базовая конфигурация сохранена
Количество кусочно-линейных признаков: 7
Alpha: 1e-05
Количество фолдов: 5
Контрольный CV MSE: 28.31085


In [326]:
def evaluate_model(
    X,
    y,
    piecewise_features=BASE_PIECEWISE_FEATURES,
    interaction_pairs=None
):
    X_array = X.to_numpy(dtype=float)
    y_array = np.asarray(y, dtype=float)

    fold_scores = []

    for train_idx, val_idx in my_kfold(
        len(y_array),
        n_splits=BASE_N_SPLITS,
        shuffle=True,
        random_state=BASE_RANDOM_STATE
    ):

        X_fold_train = X_array[train_idx]
        X_fold_val = X_array[val_idx]

        y_fold_train = y_array[train_idx]
        y_fold_val = y_array[val_idx]

        # Добавляем кусочно-линейные признаки
        train_hinges = []
        val_hinges = []

        for feature in piecewise_features:

            feature_idx = X.columns.get_loc(feature)

            train_values = X_fold_train[:, feature_idx]
            val_values = X_fold_val[:, feature_idx]

            knots = np.quantile(
                train_values,
                [0.2, 0.4, 0.6, 0.8]
            )

            train_hinges.append(
                np.column_stack([
                    np.maximum(train_values - knot, 0)
                    for knot in knots
                ])
            )

            val_hinges.append(
                np.column_stack([
                    np.maximum(val_values - knot, 0)
                    for knot in knots
                ])
            )

        X_fold_train = np.column_stack(
            [X_fold_train] + train_hinges
        )

        X_fold_val = np.column_stack(
            [X_fold_val] + val_hinges
        )

        # Нормализация по обучающему фолду
        mean = X_fold_train.mean(axis=0)
        std = X_fold_train.std(axis=0)

        std[std == 0] = 1

        X_fold_train = (X_fold_train - mean) / std
        X_fold_val = (X_fold_val - mean) / std

        # Добавляем взаимодействия, если они заданы
        if interaction_pairs is not None:

            for feature_1, feature_2 in interaction_pairs:

                idx_1 = X.columns.get_loc(feature_1)
                idx_2 = X.columns.get_loc(feature_2)

                interaction_train = (
                    X_fold_train[:, idx_1]
                    * X_fold_train[:, idx_2]
                )

                interaction_val = (
                    X_fold_val[:, idx_1]
                    * X_fold_val[:, idx_2]
                )

                interaction_mean = interaction_train.mean()
                interaction_std = interaction_train.std()

                if interaction_std == 0:
                    interaction_std = 1

                interaction_train = (
                    interaction_train - interaction_mean
                ) / interaction_std

                interaction_val = (
                    interaction_val - interaction_mean
                ) / interaction_std

                X_fold_train = np.column_stack([
                    X_fold_train,
                    interaction_train
                ])

                X_fold_val = np.column_stack([
                    X_fold_val,
                    interaction_val
                ])

        # Обучаем Ridge-регрессию
        model = MyRidgeRegression(alpha=BASE_ALPHA)

        model.fit(X_fold_train, y_fold_train)

        predictions = model.predict(X_fold_val)

        fold_mse = my_mse(y_fold_val, predictions)

        fold_scores.append(fold_mse)

    return {
        "fold_scores": fold_scores,
        "mean_mse": np.mean(fold_scores),
        "std_mse": np.std(fold_scores)
    }

In [327]:
# ==========================================
# ПРОВЕРКА ЗАФИКСИРОВАННОЙ МОДЕЛИ
# ==========================================

baseline_result = evaluate_model(
    X_train,
    y_train
)

print("MSE по фолдам:",
      np.round(baseline_result["fold_scores"], 4))

print("Средний CV MSE:",
      round(baseline_result["mean_mse"], 5))

print("Стандартное отклонение:",
      round(baseline_result["std_mse"], 5))

MSE по фолдам: [29.8669 25.779  28.464  29.7333 27.7111]
Средний CV MSE: 28.31085
Стандартное отклонение: 1.49921


In [328]:
# ==========================================
# ЭКСПЕРИМЕНТ 1
# Взаимодействие CreditScore × InterestRate
# ==========================================

experiment_result = evaluate_model(
    X_train,
    y_train,
    interaction_pairs=[
        ("CreditScore", "InterestRate")
    ]
)

print("Базовый MSE:", BASELINE_CV_MSE)

print("MSE по фолдам:",
      np.round(experiment_result["fold_scores"], 4))

print("Средний MSE:",
      round(experiment_result["mean_mse"], 5))

print("Изменение MSE:",
      round(experiment_result["mean_mse"] - BASELINE_CV_MSE, 5))

Базовый MSE: 28.31085
MSE по фолдам: [29.8588 25.7372 28.4407 29.7037 27.7503]
Средний MSE: 28.29815
Изменение MSE: -0.0127
